# Notebook 03: DeepSAM: training the surplus network

**Course:** Summer School on AI for Economics and Finance · ESOMAS, University of Torino (August 24–26, 2026)
**Session:** Day 2, 15:30 – 17:00: Practical Session on DeepHAM and Continuous-Time Models
**Slides:** `../../slides/HACT_DeepSAM_Lecture_Slides.pdf`
**Notebook role:** practical session
**Runtime:** ~1 min at `smoke`, ~8 min at `production`
**Created by:** Yucheng Yang. [Course repository](https://github.com/yangycpku/summer-school-AI-for-economics-and-finance-2026)

---


The previous two notebooks loaded a trained network. This one shows what training it means,
and — importantly — how far a short run actually gets.

Three steps:

1. **The loss.** Look at the master-equation residual on a real batch. This is the whole
   objective; there are no targets and no supervised data.
2. **Fine-tuning.** Continue training the converged checkpoint and see how little a short
   run changes it.
3. **From scratch.** Train a fresh network for the same budget and compare — which shows
   what the published training run actually bought.

In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"

## Locating the code

`src/train_nn.py` holds the whole method: the deterministic steady states, the neural
networks, the master-equation residual, the simulation of the distribution, and the training
loop. It expects to be imported with the project root as the working directory, because
`solve_steady_state` writes its output there as `.npy` files.

On Nuvolos the notebook server starts in `/files`, which mirrors the course repository, so
the cell below changes into `/files/day2/Yang/code/DeepSAM_nuvolos`. On a local clone or
Colab it steps up from `notebooks/` to the project root instead.

In [ ]:
import os
import sys
from pathlib import Path

# On Nuvolos the kernel starts in /files, which mirrors the course repository.
NUVOLOS_ROOT = "/files/day2/Yang/code/DeepSAM_nuvolos"
if os.path.isdir(NUVOLOS_ROOT):
    os.chdir(NUVOLOS_ROOT)
elif os.path.isfile("../src/train_nn.py"):
    os.chdir("..")           # a local clone or Colab: the notebook lives in notebooks/

if not (os.path.isfile("src/train_nn.py") and os.path.isfile("config/config.yaml")):
    raise FileNotFoundError(
        f"Expected the DeepSAM project root, but the working directory is "
        f"{os.getcwd()!r}. On Nuvolos that is {NUVOLOS_ROOT!r}; elsewhere, open this "
        f"notebook from inside DeepSAM_nuvolos/notebooks."
    )

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

In [ ]:
import contextlib
import io
import random
import time

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from omegaconf import OmegaConf

from train_nn import Train_NN, Master_PINN_S
import calibration_plot as calplot
import covid_shock_plot as covplot
import plotting

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CPU (no GPU visible) -- everything below still runs, more slowly")

## Choosing the run mode

The costs in this notebook are all *simulation* costs: how many paths of the economy are
simulated, and for how long. `RUN_MODE` maps onto them.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| ergodic-pool paths × horizon | 32 × 500 | 64 × 1000 | 256 × 5000 |
| pre-COVID ergodic paths | 50 | 100 | 200 |
| recovery paths averaged | 20 | 60 | 200 |
| training steps (notebook 03) | 500 | 5,000 | 20,000 |

`production` matches the settings in the replication package. Note what is *not* on this
list: training the surplus network to convergence. The full pipeline behind the shipped
checkpoint is a homotopy initialisation, a long main training phase run to a loss
threshold, and then 8 further rounds of 100,000 gradient steps with the ergodic dataset
rebuilt between rounds — several hours on an A100. That is why every notebook here starts
from the shipped checkpoint, and why notebook 03 quantifies the gap rather than trying to
close it.

In [ ]:
if RUN_MODE == "smoke":
    SIM_PATHS, SIM_T = 32, 500
    ERG_PATHS, ERG_T_END = 50, 10.0
    RECOVERY_PATHS = 20
    TRAIN_STEPS = 500
elif RUN_MODE == "teaching":
    SIM_PATHS, SIM_T = 64, 1000
    ERG_PATHS, ERG_T_END = 100, 20.0
    RECOVERY_PATHS = 60
    TRAIN_STEPS = 5_000
elif RUN_MODE == "production":
    SIM_PATHS, SIM_T = 256, 5000
    ERG_PATHS, ERG_T_END = 200, 30.0
    RECOVERY_PATHS = 200
    TRAIN_STEPS = 20_000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

print(
    f"RUN_MODE={RUN_MODE}: ergodic pool {SIM_PATHS}x{SIM_T}, "
    f"{ERG_PATHS} pre-COVID paths, {RECOVERY_PATHS} recovery paths"
)

## Calibration and the deterministic steady states

The calibration and training settings all live in `config/config.yaml`. We load it with
OmegaConf and pass it straight to `Train_NN`, so it is easy to see that the object is
simply built from that dictionary.

`solve_steady_state()` then solves the model's **deterministic** steady state once for each
aggregate state $z \in \{L, H, D\}$ — low, high, and the disaster state that stands in for
COVID. These are fixed points of the matching problem with the aggregate state frozen; they
are the anchors the aggregate-risk solution is built around, and the unemployment rates they
imply are the first thing to sanity-check against the calibration.

In [ ]:
cfg = OmegaConf.load(ROOT / "config" / "config.yaml")
params = {
    k: v for k, v in OmegaConf.to_container(cfg.train_nn, resolve=True).items()
    if k != "_target_"
}

seed = int(cfg.seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ct = Train_NN(**params)
print(f"device {ct.device} | {ct.nx} worker types x {ct.ny} firm types | output path {ct.path}")

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):     # the solver is chatty; keep the summary
    ct.solve_steady_state()
print(f"solve_steady_state: {time.monotonic() - t0:.1f}s")

gm_ss = np.load("gm_ss.npy")
gm_low = np.load("gm_low_delta.npy")
gm_high = np.load("gm_high_delta.npy")
gm_dis = np.load("gm_dis_delta.npy")

# State convention (see env.py): L is the good state (separation delta_0 - d_delta),
# H the bad state (delta_0 + d_delta), D the disaster state.
for label, gm in [("baseline", gm_ss), ("low separation (L)", gm_low),
                  ("high separation (H)", gm_high), ("disaster (D)", gm_dis)]:
    u = (ct.gw.mean() - np.mean(gm)).cpu().numpy() * 100
    print(f"  unemployment rate, {label:<22s}: {u:6.3f}%")

## The trained surplus network

The one object DeepSAM learns is the **match surplus** $S(x, y, z, g)$: the value of a match
between worker type $x$ and firm type $y$, given the aggregate state $z$ *and the entire
cross-sectional distribution* $g$ of existing matches. That last argument is the hard part —
$g$ lives in $\mathbb{R}^{n_x \times n_y}$ (55 dimensions here), which is why the network
takes it directly as an input rather than summarising it.

Everything else in the model is recovered from $S$: the acceptance sets, the vacancy
posting implied by free entry, the wage through Nash bargaining, and the drift of $g$
itself.

The checkpoint below is the converged network from the paper. Notebook 03 shows what
training it involves.

In [ ]:
pinn_S = Master_PINN_S(
    nn_width=ct.nn_width,
    nn_num_layers=ct.nn_num_layers,
    n_x=ct.nx,
    n_y=ct.ny,
).to(ct.device).float()

ckpt = torch.load(ROOT / "checkpoints" / "section3_surplus_best.pt", map_location=ct.device)
pinn_S.load_state_dict(ckpt["model_state_dict"])
pinn_S.eval()

n_par = sum(p.numel() for p in pinn_S.parameters())
print(f"Loaded the trained surplus network: {ct.nn_num_layers} layers of width "
      f"{ct.nn_width}, {n_par:,} parameters")
print(f"Input dimension: 1 (x) + 1 (y) + 1 (z) + {ct.nx * ct.ny} (g) = {3 + ct.nx * ct.ny}")

## 1. What the loss actually is

`S_pde_oper` evaluates the master-equation residual at a batch of states
$(x, y, z, g)$ drawn from the ergodic simulation. The training loss is the mean square of
that residual — there is no target and no labelled data anywhere in this method. The
network is fit to an *equation*, not to observations.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    res = ct.build_ergodic_dataloaders(
        pinn_S, N_paths=SIM_PATHS, T=SIM_T, record_interval=2,
        batch_size_train=512, num_workers=0, seed=0,
    )
train_loader, eval_loader = res["train_loader"], res["eval_loader"]
print(f"ergodic pool built in {time.monotonic() - t0:.1f}s: "
      f"{len(train_loader.dataset):,} train / {len(eval_loader.dataset):,} eval states")

S_batch = next(iter(eval_loader))[0].to(ct.device)
print(f"one batch: {tuple(S_batch.shape)}  = (batch, 1 + 1 + 1 + {ct.nx * ct.ny})")

# S_pde_oper returns (residual, acceptance probabilities, surplus). It differentiates S
# with respect to g internally, so autograd has to be live even though we never call
# backward() here.
with torch.enable_grad():
    resid, alphas, S_vals = ct.S_pde_oper(pinn_S, S_batch)

print(f"surplus S on this batch : mean {float(S_vals.mean()):+.4f}, "
      f"range [{float(S_vals.min()):+.4f}, {float(S_vals.max()):+.4f}]")
print(f"master-equation residual: mean square {float((resid ** 2).mean()):.3e}")
print("That mean square is the training loss -- there is no target anywhere in it.")

## 2. Fine-tuning the converged network

Continue training the checkpoint at a low learning rate. The exact loss level depends on
the simulated evaluation pool, but the shape is what matters: it starts orders of
magnitude below the from-scratch run in the next section and moves comparatively little.

In [ ]:
model_ft = Master_PINN_S(
    nn_width=ct.nn_width, nn_num_layers=ct.nn_num_layers, n_x=ct.nx, n_y=ct.ny
).to(ct.device).float()
model_ft.load_state_dict(ckpt["model_state_dict"])

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    out_ft = ct.train_with_ergodic_loaders(
        model_ft, optim.Adam(model_ft.parameters(), lr=1e-5),
        train_loader, eval_loader,
        total_steps=TRAIN_STEPS, eval_every=max(1, TRAIN_STEPS // 5),
        print_every=max(1, TRAIN_STEPS // 5), max_eval_batches=4,
    )
print(f"{TRAIN_STEPS} fine-tuning steps in {time.monotonic() - t0:.1f}s "
      f"({(time.monotonic() - t0) / TRAIN_STEPS * 1000:.0f} ms/step)")
print(f"best eval loss: {out_ft['best_eval_loss']:.3e}")

## 3. The same budget, from scratch

Now a fresh network. It first gets the cheap **homotopy initialisation** — fitted to a
simple analytic guess, just to put it in a sensible region — and then the same number of
gradient steps as the fine-tuning run above.

In [ ]:
model_new = Master_PINN_S(
    nn_width=ct.nn_width, nn_num_layers=ct.nn_num_layers, n_x=ct.nx, n_y=ct.ny
).to(ct.device).float()

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    ct.initial_guess(model_new, optim.Adam(model_new.parameters(), lr=ct.lr_init),
                     epochs=min(2000, 4 * TRAIN_STEPS), option="S")
    out_new = ct.train_with_ergodic_loaders(
        model_new, optim.Adam(model_new.parameters(), lr=1e-4),
        train_loader, eval_loader,
        total_steps=TRAIN_STEPS, eval_every=max(1, TRAIN_STEPS // 5),
        print_every=max(1, TRAIN_STEPS // 5), max_eval_batches=4,
    )
print(f"from scratch, {TRAIN_STEPS} steps in {time.monotonic() - t0:.1f}s")
print(f"best eval loss: {out_new['best_eval_loss']:.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=140)
for out, label, style in [(out_ft, "continued from the checkpoint", "-"),
                          (out_new, f"from scratch ({TRAIN_STEPS} steps)", "--")]:
    log = out["eval_log"]
    ax.semilogy(log["steps"], log["loss"], style, linewidth=2, marker="o",
                markersize=4, label=label)
ax.set_xlabel("gradient step")
ax.set_ylabel("evaluation loss (master-equation residual)")
ax.set_title("What a short training budget buys")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gap = out_new["best_eval_loss"] / out_ft["best_eval_loss"]
print(f"after {TRAIN_STEPS} steps the fresh network is {gap:,.0f}x further from solving "
      f"the master equation than the shipped checkpoint")
print("The published pipeline: homotopy initialisation, a main training phase to a loss "
      "threshold, then 8 rounds of 100,000 steps on rebuilt ergodic datasets.")
assert np.isfinite(gap) and gap > 1.0, "the fresh network should not beat the checkpoint"

## Summary

* The objective is a **residual**, not a fit to data: DeepSAM trains the surplus network to
  satisfy the master equation on states the simulated economy actually visits.
* Training and simulation are interleaved — the dataset is rebuilt under the current network,
  because the ergodic distribution itself depends on the solution.
* A short run gets the loss falling quickly and then stalls orders of magnitude short. The
  last two digits of accuracy are most of the compute.

## Takeaway

This is the practical shape of deep-learning solution methods for continuous-time models:
cheap to get something plausible, expensive to get something you would put in a paper. Which
is why the checkpoint is shipped, and why the notebooks that produce the *economics* —
01 and 02 — never train anything.